# What Is an Agent? A Hands-On Demonstration

## Understanding AI Agents

An **AI agent** is a system that can:
1. **Perceive** its environment (through tools and data)
2. **Reason** about what it observes (using an LLM)
3. **Act** to achieve goals (by calling tools)
4. **Learn** from feedback (through context management)

### Core Components We'll Build Today

**🔧 Tools**: Functions the agent can call to interact with data
- File discovery and data loading
- SQL queries for analysis  
- Visualization generation
- Statistical computations

**📊 State Management**: Tracking what the agent knows
- Available data tables and schemas
- Completed operations
- Error recovery information

**🧠 Planning & Reasoning**: How the agent approaches problems
- Breaking down complex tasks into steps
- Choosing appropriate tools
- Recovering from errors

**💬 Context Management**: Maintaining conversation coherence
- Structured prompts and responses
- Token usage optimization
- Progressive information disclosure

### Today's Challenge

We'll build a financial analyst agent that can:
- **Discover** the right data file among multiple options
- **Analyze** daily sales patterns throughout the week
- **Identify** and adjust for anomalies
- **Visualize** findings with appropriate charts
- **Report** insights in a professional manner

The agent won't be told which file to use or its structure - it must figure this out itself!

## 1. Setup and Dependencies

First, we'll install required packages and set up our environment.

In [ ]:
# Install required packages
!pip install -q openai pandas numpy duckdb matplotlib python-dotenv

In [ ]:
# Import all necessary libraries
import os
import re
import json
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Tuple
from datetime import datetime, timedelta
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path

# Configure matplotlib for inline display
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.style.use('seaborn-v0_8-darkgrid')

## 2. API Configuration

Configure the OpenAI API client. The agent will use GPT-5 Mini for reasoning.

In [ ]:
# Load environment variables from .env file
load_dotenv()

# Read API key from environment
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError(
        "Missing OPENAI_API_KEY environment variable!\n"
        "Please set it in .env file or environment:\n"
        "export OPENAI_API_KEY='your-key-here'"
    )

# Create OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Agent configuration
MODEL = "gpt-5-mini"  # The LLM that powers our agent's reasoning
MAX_AGENT_STEPS = 20   # Maximum reasoning steps before timeout

print(f"✅ Agent configured with {MODEL}")
print(f"   Max steps: {MAX_AGENT_STEPS}")
print("   Note: Step limits are for demo purposes. In production, we'd analyze")
print("         typical task completion patterns to set appropriate limits.")

## 3. Tool Infrastructure

Tools are the agent's way of interacting with the world. We'll build a flexible system for defining and calling tools.

### Key Concepts:
- **Tool Definition**: Name, description, parameters, and implementation
- **Tool Registry**: Central management of available tools
- **Tool Calling**: Custom XML-based format for transparency

In [ ]:
@dataclass
class Tool:
    """Defines a tool the agent can use.
    
    Each tool has:
    - name: How the agent refers to it
    - description: What it does (helps agent choose the right tool)
    - parameters: What inputs it needs
    - func: The actual Python function to execute
    - examples: Usage examples for the agent
    """
    name: str
    description: str
    parameters: List[Dict[str, str]]
    func: Callable
    examples: List[str] = field(default_factory=list)


class ToolRegistry:
    """Manages all available tools for the agent.
    
    The registry:
    - Stores tool definitions
    - Generates instructions for the LLM
    - Executes tool calls safely
    """
    def __init__(self):
        self.tools = {}
    
    def register(self, tool: Tool):
        """Register a new tool."""
        self.tools[tool.name] = tool
    
    def get_instructions(self) -> str:
        """Generate formatted instructions for all tools.
        This becomes part of the agent's system prompt.
        """
        instructions = ["Available tools:\n"]
        for tool in self.tools.values():
            params_str = ", ".join([f"{p['name']}: {p['type']}" for p in tool.parameters])
            instructions.append(f"- {tool.name}({params_str}): {tool.description}")
            if tool.examples:
                instructions.append(f"  Example: {tool.examples[0]}")
        return "\n".join(instructions)
    
    def call(self, tool_name: str, **kwargs) -> Any:
        """Execute a tool by name with given parameters."""
        if tool_name not in self.tools:
            raise ValueError(f"Tool '{tool_name}' not found")
        return self.tools[tool_name].func(**kwargs)


def parse_tool_calls(text: str) -> List[Dict[str, Any]]:
    """Parse tool calls from agent's response.
    
    We use XML tags for tool calls: <tool>tool_name(param="value")</tool>
    This format is easy for LLMs to generate and for us to parse.
    """
    tool_calls = []
    
    # Find all <tool>...</tool> blocks
    pattern = r'<tool>(.*?)</tool>'
    matches = re.findall(pattern, text, re.DOTALL)
    
    for match in matches:
        match = match.strip()
        
        # Parse: tool_name(param1="value1", param2="value2")
        call_pattern = r'^(\w+)\s*\((.*)\)$'
        call_match = re.match(call_pattern, match, re.DOTALL)
        
        if not call_match:
            continue
        
        tool_name = call_match.group(1)
        params_str = call_match.group(2).strip()
        
        # Parse parameters
        params = {}
        if params_str:
            param_pattern = r'(\w+)\s*=\s*(?:"([^"\\]*(?:\\.[^"\\]*)*)"|([^,\)]+))'
            param_matches = re.findall(param_pattern, params_str)
            
            for param_name, quoted_value, unquoted_value in param_matches:
                param_value = quoted_value if quoted_value else unquoted_value.strip()
                param_value = param_value.replace('\\"', '"').replace('\\n', '\n')
                
                # Type conversion
                if param_value.lower() in ['true', 'false']:
                    params[param_name] = param_value.lower() == 'true'
                elif param_value.isdigit():
                    params[param_name] = int(param_value)
                else:
                    try:
                        params[param_name] = float(param_value)
                    except:
                        params[param_name] = param_value
        
        tool_calls.append({
            'name': tool_name,
            'parameters': params
        })
    
    return tool_calls


# Global storage for data tables the agent creates
TABLES = {}

## 4. Agent Tools Implementation

Now we'll implement the specific tools our financial analyst agent needs. Each tool serves a specific purpose in the agent's workflow.

In [ ]:
# Tool 1: File Discovery
def list_data_files(folder: str = "data") -> Dict:
    """List all CSV files in the specified folder.
    This helps the agent discover available data sources.
    """
    files = []
    folder_path = Path(folder)
    
    if not folder_path.exists():
        return {"error": f"Folder '{folder}' not found"}
    
    for file_path in folder_path.glob("*.csv"):
        size_kb = file_path.stat().st_size / 1024
        files.append({
            "name": file_path.name,
            "size_kb": round(size_kb, 1),
            "path": str(file_path)
        })
    
    return {
        "folder": folder,
        "csv_files": files,
        "count": len(files)
    }


# Tool 2: Load CSV Data
def load_csv(path: str, name: str) -> Dict:
    """Load a CSV file into memory as a table.
    The agent can then query this table with SQL.
    """
    try:
        df = pd.read_csv(path)
        TABLES[name] = df
        
        # Save to workings folder for inspection
        os.makedirs("workings", exist_ok=True)
        df.to_csv(f"workings/{name}_loaded.csv", index=False)
        
        return {
            "loaded_rows": len(df),
            "columns": list(df.columns),
            "table_name": name,
            "sample": df.head(3).to_dict('records')
        }
    except Exception as e:
        return {"error": str(e)}


# Tool 3: Examine Table Schema
def examine_table(name: str) -> Dict:
    """Examine a loaded table's structure and content.
    Helps the agent understand what data is available.
    """
    if name not in TABLES:
        return {"error": f"Table '{name}' not found"}
    
    df = TABLES[name]
    schema = {}
    
    for col in df.columns:
        dtype = str(df[col].dtype)
        
        # Infer semantic type
        sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else None
        if 'datetime' in dtype or (isinstance(sample, str) and 'T' in str(sample)):
            semantic_type = 'timestamp'
        elif pd.api.types.is_numeric_dtype(df[col]):
            semantic_type = 'numeric'
        else:
            semantic_type = 'categorical'
        
        schema[col] = {
            "dtype": dtype,
            "semantic_type": semantic_type,
            "non_null": int(df[col].notna().sum()),
            "unique_values": int(df[col].nunique()),
            "sample_values": df[col].dropna().head(5).tolist()
        }
    
    return {
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "schema": schema
    }


# Tool 4: SQL Query Engine
def sql_query(query: str, save_as: Optional[str] = None) -> Dict:
    """Execute SQL queries using DuckDB.
    This is the agent's primary analysis tool.
    """
    try:
        conn = duckdb.connect(':memory:')
        
        # Register all tables
        for table_name, df in TABLES.items():
            conn.register(table_name, df)
        
        # Execute query
        result_df = conn.execute(query).fetchdf()
        
        # Save result if requested
        if save_as:
            TABLES[save_as] = result_df
            # Also save to file
            result_df.to_csv(f"workings/{save_as}.csv", index=False)
        
        conn.close()
        
        return {
            "rows": len(result_df),
            "columns": list(result_df.columns),
            "saved_as": save_as,
            "result": result_df.head(10).to_dict('records')
        }
    except Exception as e:
        return {"error": str(e)}


# Tool 5: Statistical Analysis
def analyze_statistics(table: str, column: str, group_by: Optional[str] = None) -> Dict:
    """Compute statistical summaries.
    Helps the agent understand data distributions.
    """
    if table not in TABLES:
        return {"error": f"Table '{table}' not found"}
    
    df = TABLES[table]
    
    if column not in df.columns:
        return {"error": f"Column '{column}' not found"}
    
    if group_by and group_by in df.columns:
        # Grouped statistics
        stats = df.groupby(group_by)[column].agg([
            'count', 'mean', 'std', 'min', 'max'
        ]).round(2).to_dict('index')
    else:
        # Overall statistics
        stats = {
            "count": int(df[column].count()),
            "mean": float(df[column].mean()),
            "std": float(df[column].std()),
            "min": float(df[column].min()),
            "max": float(df[column].max()),
            "q25": float(df[column].quantile(0.25)),
            "median": float(df[column].quantile(0.5)),
            "q75": float(df[column].quantile(0.75))
        }
    
    return {
        "table": table,
        "column": column,
        "group_by": group_by,
        "statistics": stats
    }


# Tool 6: Flexible Visualization
def create_chart(
    chart_type: str,
    table: str,
    x: Optional[str] = None,
    y: Optional[str] = None,
    title: str = "",
    xlabel: str = "",
    ylabel: str = "",
    color: Optional[str] = None
) -> str:
    """Create various types of charts using matplotlib.
    The agent can create bar, line, scatter, and histogram charts.
    """
    if table not in TABLES:
        return f"Error: Table '{table}' not found"
    
    df = TABLES[table]
    
    plt.figure(figsize=(12, 7))
    
    try:
        if chart_type == "bar":
            if color and color in df.columns:
                for group in df[color].unique():
                    subset = df[df[color] == group]
                    plt.bar(subset[x], subset[y], label=group, alpha=0.8)
                plt.legend()
            else:
                plt.bar(df[x], df[y], color='steelblue', alpha=0.8)
        
        elif chart_type == "line":
            if color and color in df.columns:
                for group in df[color].unique():
                    subset = df[df[color] == group]
                    plt.plot(subset[x], subset[y], marker='o', label=group, linewidth=2)
                plt.legend()
            else:
                plt.plot(df[x], df[y], marker='o', color='darkblue', linewidth=2)
        
        elif chart_type == "scatter":
            if color and color in df.columns:
                for group in df[color].unique():
                    subset = df[df[color] == group]
                    plt.scatter(subset[x], subset[y], label=group, alpha=0.7, s=50)
                plt.legend()
            else:
                plt.scatter(df[x], df[y], alpha=0.7, s=50, color='navy')
        
        elif chart_type == "histogram":
            plt.hist(df[x] if x else df[y], bins=30, alpha=0.7, color='teal', edgecolor='black')
        
        else:
            return f"Error: Unknown chart type '{chart_type}'"
        
        plt.title(title or f"{chart_type.title()} Chart", fontsize=14, fontweight='bold')
        plt.xlabel(xlabel or x, fontsize=12)
        plt.ylabel(ylabel or y, fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        # Save the figure
        filename = f"workings/{title.replace(' ', '_').lower()}.png"
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        plt.show()
        
        return f"Chart created and saved as {filename}"
        
    except Exception as e:
        return f"Error creating chart: {str(e)}"

## 5. Register Tools

Now we register all tools with the registry, making them available to the agent.

In [ ]:
# Create the tool registry
registry = ToolRegistry()

# Register file discovery tool
registry.register(Tool(
    name="list_data_files",
    description="List all CSV files in a folder to discover available data",
    parameters=[
        {"name": "folder", "type": "str", "description": "Folder to search (default: 'data')"}
    ],
    func=list_data_files,
    examples=['<tool>list_data_files(folder="data")</tool>']
))

# Register data loading tool
registry.register(Tool(
    name="load_csv",
    description="Load a CSV file into memory as a queryable table",
    parameters=[
        {"name": "path", "type": "str", "description": "Path to the CSV file"},
        {"name": "name", "type": "str", "description": "Name for the loaded table"}
    ],
    func=load_csv,
    examples=['<tool>load_csv(path="data/sales.csv", name="sales")</tool>']
))

# Register table examination tool
registry.register(Tool(
    name="examine_table",
    description="Examine a table's structure, data types, and sample values",
    parameters=[
        {"name": "name", "type": "str", "description": "Name of the table to examine"}
    ],
    func=examine_table,
    examples=['<tool>examine_table(name="sales")</tool>']
))

# Register SQL query tool
registry.register(Tool(
    name="sql_query",
    description="Execute SQL queries on loaded tables using DuckDB",
    parameters=[
        {"name": "query", "type": "str", "description": "SQL query to execute"},
        {"name": "save_as", "type": "str", "description": "Optional name to save results"}
    ],
    func=sql_query,
    examples=['<tool>sql_query(query="SELECT * FROM sales LIMIT 5", save_as="sample")</tool>']
))

# Register statistics tool
registry.register(Tool(
    name="analyze_statistics",
    description="Compute statistical summaries for numeric columns",
    parameters=[
        {"name": "table", "type": "str", "description": "Table name"},
        {"name": "column", "type": "str", "description": "Column to analyze"},
        {"name": "group_by", "type": "str", "description": "Optional grouping column"}
    ],
    func=analyze_statistics,
    examples=['<tool>analyze_statistics(table="sales", column="amount", group_by="region")</tool>']
))

# Register visualization tool
registry.register(Tool(
    name="create_chart",
    description="Create bar, line, scatter, or histogram charts",
    parameters=[
        {"name": "chart_type", "type": "str", "description": "Type: bar, line, scatter, histogram"},
        {"name": "table", "type": "str", "description": "Table with data"},
        {"name": "x", "type": "str", "description": "X-axis column"},
        {"name": "y", "type": "str", "description": "Y-axis column"},
        {"name": "title", "type": "str", "description": "Chart title"},
        {"name": "xlabel", "type": "str", "description": "X-axis label"},
        {"name": "ylabel", "type": "str", "description": "Y-axis label"},
        {"name": "color", "type": "str", "description": "Column for color grouping"}
    ],
    func=create_chart,
    examples=['<tool>create_chart(chart_type="bar", table="weekly", x="day", y="revenue", title="Revenue by Day")</tool>']
))

print("📚 Tool Registry Loaded")
print("=" * 50)
print(registry.get_instructions())

## 6. Context Management System

Good context management is crucial for agent performance. Our system tracks state, provides error recovery, and helps the agent stay aware of its progress.

### Key Features:
- **State Tracking**: Always knows what tables and columns exist
- **Error Recovery**: Provides hints when operations fail
- **Step Awareness**: Tracks remaining steps to ensure task completion
- **Structured Formatting**: Clear, consistent response formatting

**Note on Step Limits**: We use a maximum step limit for demonstration purposes. In production, you would:
- Analyze typical task completion patterns to determine appropriate limits
- Implement dynamic limits based on task complexity
- Use timeout mechanisms rather than hard step counts
- Monitor and adjust based on real-world performance data

In [ ]:
class ContextManager:
    """Manages the agent's context window and state.
    
    This is critical for agent performance - it ensures the agent:
    - Always knows what resources are available
    - Gets helpful hints when errors occur
    - Knows how many steps remain
    - Receives well-structured information
    """
    
    def __init__(self, max_context_tokens: int = 8000):
        self.max_tokens = max_context_tokens
        self.state_summary = {}
        self.recent_errors = []
        self.current_step = 0
        self.max_steps = MAX_AGENT_STEPS
        
    def update_state(self):
        """Update the current state summary.
        Tracks all tables and their schemas.
        """
        self.state_summary = {
            "tables": {name: {"rows": len(df), "columns": list(df.columns)} 
                      for name, df in TABLES.items()},
            "recent_errors": self.recent_errors[-3:] if self.recent_errors else []
        }
    
    def set_step(self, step: int):
        """Update the current step number."""
        self.current_step = step
    
    def add_error(self, tool_name: str, error: str):
        """Track errors with recovery hints."""
        error_entry = {
            "tool": tool_name,
            "error": error,  # No truncation - full error for context
            "hint": self._get_error_hint(error)
        }
        self.recent_errors.append(error_entry)
        if len(self.recent_errors) > 5:
            self.recent_errors.pop(0)
    
    def _get_error_hint(self, error: str) -> str:
        """Generate helpful hints based on error type."""
        error_lower = error.lower()
        
        if "not found" in error_lower or "catalog error" in error_lower:
            return "Check available tables with examine_table"
        elif "column" in error_lower or "binder error" in error_lower:
            return "Use examine_table to see correct column names"
        elif "syntax" in error_lower:
            return "Check SQL syntax - ensure proper quotes and formatting"
        elif "permission" in error_lower:
            return "Check file path and permissions"
        return "Review the error message and adjust your approach"
    
    def get_state_context(self) -> str:
        """Generate formatted state summary for the agent."""
        lines = ["### Current State:"]
        
        # Add step tracking
        remaining_steps = self.max_steps - self.current_step
        lines.append(f"\n**Progress: Step {self.current_step}/{self.max_steps}** ({remaining_steps} steps remaining)")
        if remaining_steps <= 5:
            lines.append("⚠️ **Warning**: Only a few steps remaining. Prioritize completing your analysis.")
        
        if self.state_summary.get("tables"):
            lines.append("\n**Available Tables:**")
            for name, info in self.state_summary["tables"].items():
                cols = ", ".join(info["columns"][:5])
                if len(info["columns"]) > 5:
                    cols += f", ... ({len(info['columns'])-5} more)"
                lines.append(f"- `{name}`: {info['rows']} rows | Columns: {cols}")
        else:
            lines.append("\n**No tables loaded yet**")
        
        if self.state_summary.get("recent_errors"):
            lines.append("\n**Recent Errors (with hints):**")
            for err in self.state_summary["recent_errors"]:
                lines.append(f"- {err['tool']}: {err['error']}")
                lines.append(f"  💡 Hint: {err['hint']}")
        
        return "\n".join(lines)
    
    def format_tool_results(self, tool_results: list) -> str:
        """Format tool execution results clearly."""
        formatted = ["### Tool Execution Results:\n"]
        
        success_count = sum(1 for r in tool_results if not "Error" in r)
        error_count = sum(1 for r in tool_results if "Error" in r)
        
        formatted.append(f"✅ Successful: {success_count} | ❌ Failed: {error_count}\n")
        
        for result in tool_results:
            if "Error" in result:
                formatted.append(f"• ❌ {result}")
            else:
                # No truncation - show full results
                formatted.append(f"• {result}")
        
        return "\n".join(formatted)

## 7. Trace Management

The trace system records every step of the agent's reasoning process. This is invaluable for debugging and understanding how the agent solves problems.

In [ ]:
@dataclass
class TraceEntry:
    """A single entry in the agent's trace log."""
    timestamp: str
    step: int
    type: str  # plan, thought, action, observation, final
    content: Any


class Trace:
    """Records the agent's entire problem-solving process.
    
    The trace captures:
    - Planning steps
    - Tool calls and results
    - Reasoning thoughts
    - Final answers
    """
    def __init__(self):
        self.entries = []
        self.step_counter = 0
    
    def add(self, entry_type: str, content: Any):
        """Add a new entry to the trace."""
        self.step_counter += 1
        entry = TraceEntry(
            timestamp=datetime.now().isoformat(),
            step=self.step_counter,
            type=entry_type,
            content=content
        )
        self.entries.append(entry)
    
    def save_to_file(self, filename: str = "workings/agent_trace.txt"):
        """Save the complete trace to a text file."""
        os.makedirs(os.path.dirname(filename), exist_ok=True)
        
        with open(filename, 'w') as f:
            f.write("AGENT EXECUTION TRACE\n")
            f.write("=" * 60 + "\n\n")
            
            for entry in self.entries:
                f.write(f"[Step {entry.step}] {entry.timestamp}\n")
                f.write(f"Type: {entry.type.upper()}\n")
                f.write("-" * 40 + "\n")
                
                if isinstance(entry.content, str):
                    f.write(entry.content)
                else:
                    f.write(json.dumps(entry.content, indent=2, default=str))
                
                f.write("\n\n" + "=" * 60 + "\n\n")
        
        return filename
    
    def get_summary(self) -> Dict:
        """Get a summary of the trace."""
        return {
            "total_steps": len(self.entries),
            "tool_calls": sum(1 for e in self.entries if e.type == "action"),
            "errors": sum(1 for e in self.entries if "error" in str(e.content).lower()),
            "duration": (
                datetime.fromisoformat(self.entries[-1].timestamp) - 
                datetime.fromisoformat(self.entries[0].timestamp)
            ).total_seconds() if len(self.entries) > 1 else 0
        }

## 8. The Agent Loop

This is the heart of our agent - the main loop that orchestrates reasoning, tool calling, and context management.

### How It Works:
1. **Initialize** with system prompt and task
2. **Plan** the approach to solving the problem
3. **Execute** tools based on the plan
4. **Update** context after each action
5. **Iterate** until task is complete or max steps reached

In [ ]:
def create_system_prompt(registry: ToolRegistry) -> str:
    """Create the system prompt that defines the agent's behavior.
    
    This prompt:
    - Establishes the agent's persona
    - Explains available tools
    - Sets expectations for output format
    """
    return f"""You are a senior financial analyst with expertise in data analysis and visualization.
Your role is to analyze business data, identify patterns, and provide actionable insights.

## Your Approach:
1. First, discover and understand available data
2. Create a clear analysis plan
3. Execute analysis using appropriate tools
4. Visualize findings effectively
5. Provide professional insights

## Tool Usage:
Use XML tags for tool calls: <tool>tool_name(param1="value1", param2="value2")</tool>

{registry.get_instructions()}

## Guidelines:
- Start by discovering available data files
- Examine data structure before analysis
- Use SQL for data transformations
- Create clear visualizations
- Provide specific, quantified insights
- Save intermediate results for reproducibility
- Be mindful of your step count and complete your analysis efficiently
"""


def run_agent(goal: str, max_steps: int = MAX_AGENT_STEPS) -> Tuple[str, Trace]:
    """Run the agent to solve a given goal.
    
    This is the main agent loop that:
    - Manages conversation with the LLM
    - Executes tool calls
    - Maintains context with step tracking
    - Records trace
    """
    import time
    
    # Initialize components
    trace = Trace()
    context_mgr = ContextManager()
    context_mgr.max_steps = max_steps
    system_prompt = create_system_prompt(registry)
    
    print(f"🔄 Initializing Financial Analyst Agent")
    print(f"   Model: {MODEL}")
    print(f"   Max steps: {max_steps}")
    print("\n" + "=" * 60 + "\n")
    
    # Initialize conversation
    context_mgr.update_state()
    initial_context = context_mgr.get_state_context()
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Task: {goal}\n\n{initial_context}"}
    ]
    
    step = 0
    
    while step < max_steps:
        step += 1
        context_mgr.set_step(step)
        print(f"📍 Step {step}/{max_steps}: ", end="", flush=True)
        
        try:
            # Call the LLM
            start_time = time.time()
            
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                max_completion_tokens=4000,
                timeout=120
            )
            
            elapsed = time.time() - start_time
            content = response.choices[0].message.content
            
            print(f"Thinking... ✓ ({elapsed:.1f}s)")
            
            # Add response to trace
            trace.add("thought", content)
            
        except Exception as e:
            print(f" ✗ Error!")
            print(f"❌ API Error: {e}")
            trace.add("error", str(e))
            break
        
        # Parse and execute tool calls
        tool_calls = parse_tool_calls(content)
        
        if tool_calls:
            print(f"    🔧 Executing {len(tool_calls)} tool(s):")
            
            tool_results = []
            for i, tool_call in enumerate(tool_calls, 1):
                tool_name = tool_call['name']
                params = tool_call['parameters']
                
                print(f"       {i}. {tool_name}: ", end="", flush=True)
                
                # Record action
                trace.add("action", {"tool": tool_name, "params": params})
                
                try:
                    result = registry.call(tool_name, **params)
                    tool_results.append(f"Result of {tool_name}: {json.dumps(result, default=str)}")
                    trace.add("observation", result)
                    print("✓")
                    
                except Exception as e:
                    error_msg = f"Error in {tool_name}: {str(e)}"
                    tool_results.append(error_msg)
                    trace.add("observation", {"error": str(e)})
                    context_mgr.add_error(tool_name, str(e))
                    print("✗")
            
            # Update state and format results
            context_mgr.update_state()
            messages.append({"role": "assistant", "content": content})
            
            formatted_results = context_mgr.format_tool_results(tool_results)
            state_context = context_mgr.get_state_context()
            
            # Add recovery hints if there were errors
            recovery_hint = ""
            if any("Error" in r for r in tool_results):
                recovery_hint = "\n\n💡 **Recovery Tip:** Check the state above and use examine_table if needed."
            
            messages.append({"role": "user", "content": f"{formatted_results}\n\n{state_context}{recovery_hint}\n\nContinue with your analysis."})
        
        # Check for final answer
        elif any(keyword in content.lower() for keyword in ["final", "conclusion", "summary", "insights:"]):
            print("    🎯 Analysis complete")
            trace.add("final", content)
            return content, trace
        
        else:
            # Continue conversation
            messages.append({"role": "assistant", "content": content})
            context_mgr.update_state()
            state_context = context_mgr.get_state_context()
            messages.append({"role": "user", "content": f"{state_context}\n\nPlease continue your analysis using the available tools."})
    
    print("\n⚠️ Max steps reached")
    return "Analysis incomplete - max steps reached.", trace


def summarize_for_model(obj: Any, max_length: int = 500) -> str:
    """Summarize objects for the LLM context."""
    if isinstance(obj, pd.DataFrame):
        return f"DataFrame({len(obj)} rows, {len(obj.columns)} cols)\n{obj.head(3).to_csv()}"
    elif isinstance(obj, (dict, list)):
        s = json.dumps(obj, indent=2, default=str)
        return s[:max_length] + "..." if len(s) > max_length else s
    else:
        s = str(obj)
        return s[:max_length] + "..." if len(s) > max_length else s

## 9. Run the Agent

Now let's see our financial analyst agent in action! We'll give it a challenging task that requires:
- Finding the right data file
- Understanding the data structure
- Performing complex analysis
- Creating visualizations
- Providing insights

In [ ]:
# Clear any previous state
TABLES.clear()
os.makedirs("workings", exist_ok=True)

# Define the analysis task
goal = """Analyze daily sales patterns throughout the week. 
Find which days perform best and worst, identify any anomalies or unusual patterns, 
and provide visualizations to support your findings. 
Consider factors like day-of-week effects and any outliers in the data, e.g. public holidays, big sale events.
Provide 3-5 specific insights with quantified impacts."""

print("🚀 Starting Financial Analyst Agent\n")
print("Task:", goal)
print("\n" + "=" * 60 + "\n")

# Run the agent with 20 steps max
try:
    final_answer, trace = run_agent(goal, max_steps=20)
    
    print("\n" + "=" * 60)
    print("✅ ANALYSIS COMPLETE")
    print("=" * 60)
    print("\n### Final Report:\n")
    print(final_answer)
    
    # Save trace to file
    trace_file = trace.save_to_file()
    print(f"\n📄 Full trace saved to: {trace_file}")
    
    # Show summary statistics
    summary = trace.get_summary()
    print(f"\n📊 Execution Summary:")
    print(f"   Total steps: {summary['total_steps']}")
    print(f"   Tool calls: {summary['tool_calls']}")
    print(f"   Errors encountered: {summary['errors']}")
    print(f"   Duration: {summary['duration']:.1f} seconds")
    
    # List created files
    print(f"\n📁 Files created in workings folder:")
    for file in os.listdir("workings"):
        file_path = os.path.join("workings", file)
        size = os.path.getsize(file_path) / 1024
        print(f"   - {file}: {size:.1f} KB")
    
except Exception as e:
    print(f"\n❌ Agent failed: {e}")
    import traceback
    traceback.print_exc()

## 10. Key Takeaways

### What Makes an Agent?

Through this demonstration, we've seen that an AI agent is more than just an LLM - it's a complete system with:

1. **Tools for Action**: The agent interacts with its environment through well-defined tools
2. **State Management**: It tracks what it knows and what resources are available
3. **Planning Capability**: It breaks down complex tasks into manageable steps
4. **Error Recovery**: It learns from failures and adjusts its approach
5. **Context Awareness**: It maintains coherent conversation through structured context

### Design Principles

**Transparency**: Using XML tags for tool calls makes the agent's actions clear and debuggable

**Modularity**: Each tool is independent, making it easy to add new capabilities

**State Tracking**: Always knowing what data is available prevents confusion

**Error Resilience**: Providing hints helps the agent recover from mistakes

**Traceability**: Recording every step enables debugging and improvement

### Real-World Applications

This pattern can be extended to:
- **Customer Service**: Agents that can query databases and update records
- **DevOps**: Agents that monitor systems and respond to incidents
- **Research**: Agents that gather information and synthesize reports
- **Data Analysis**: Agents that explore datasets and find insights

### Next Steps

To build production agents, consider:
- **Async Execution**: Run tools in parallel for better performance
- **Caching**: Store results to avoid repeated computations
- **Safety**: Implement guardrails and validation
- **Monitoring**: Track performance and errors in production
- **Scaling**: Use queues and workers for high-volume processing